<a href="https://colab.research.google.com/github/adenikeadewumi/Python-programming-for-ML-WIEOAU/blob/main/10_numpy/10_numpy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 10 — NumPy: Numerical Computing

**Learning Objectives:** Arrays, indexing, broadcasting, linear algebra, random numbers

**Estimated time:** 60–75 minutes

---

## 10.1 What Is NumPy and Why Does It Exist?

**The problem with Python lists for maths:**
Python lists are flexible — they can hold any types, grow dynamically, and are easy to use. But this flexibility comes at a cost. When you add two lists, Python has to:
1. Check the type of each element individually
2. Do the operation one element at a time
3. Build a brand new list to hold the results

For 10 elements this is fine. For 10 million elements (a typical dataset), it is 10–100× too slow.

**What NumPy does differently:**
NumPy arrays store all elements as the same type (e.g. all 64-bit floats) in a contiguous block of memory. This allows NumPy to hand off the computation to pre-compiled C and Fortran code, which operates on the entire array at once. The result: operations that take seconds in pure Python take milliseconds in NumPy.

**Why you must learn NumPy:**
- **Pandas** is built on NumPy arrays
- **Scikit-learn** takes NumPy arrays as input and returns them as output
- **TensorFlow and PyTorch** tensors behave almost identically to NumPy arrays
- Understanding NumPy means you understand the data format that ALL ML tools use

**The golden rule of NumPy:**
Avoid Python loops over array elements at all costs. Use NumPy's built-in operations instead. This is called **vectorisation**.

In [ ]:
import numpy as np

# Speed comparison: Python list vs NumPy array
import time

size = 1_000_000
py_list = list(range(size))
np_array = np.arange(size)

# Python list: loop through and square each element
start = time.time()
py_result = [x**2 for x in py_list]
py_time = time.time() - start

# NumPy: vectorised operation — no loop needed
start = time.time()
np_result = np_array ** 2
np_time = time.time() - start

print(f"Python list:  {py_time:.4f}s")
print(f"NumPy array:  {np_time:.4f}s")
print(f"NumPy is {py_time/np_time:.0f}x faster")

## 10.2 Creating Arrays

**The core object in NumPy is the `ndarray`** (n-dimensional array). Think of it as a grid that can be 1D (a vector), 2D (a matrix), 3D (a cube), or higher.

**Key properties every array has:**
- `.shape` — a tuple giving the size in each dimension, e.g. `(3, 4)` means 3 rows, 4 columns
- `.dtype` — the data type of each element, e.g. `float64`, `int32`
- `.ndim` — the number of dimensions
- `.size` — the total number of elements

**Choosing the right dtype matters in ML:**
`float32` uses half the memory of `float64` and is faster on GPUs. Most deep learning uses `float32`.

In [ ]:
import numpy as np

# From a Python list — most basic creation method
a = np.array([1, 2, 3, 4, 5])
b = np.array([[1, 2, 3],
              [4, 5, 6]])   # 2D array

print("1D array:", a)
print("Shape:", a.shape)   # (5,) — one dimension of size 5
print()
print("2D array:")
print(b)
print("Shape:", b.shape)   # (2, 3) — 2 rows, 3 columns
print("Dtype:", b.dtype)
print("Ndim:", b.ndim)
print("Size:", b.size)     # total elements: 2*3 = 6

In [ ]:
# NumPy provides many functions to create arrays without typing every value

print("zeros — all zeros, useful as a starting point:")
print(np.zeros((3, 4)))

print("
ones — all ones:")
print(np.ones((2, 3)))

print("
eye — identity matrix (1s on diagonal, 0s elsewhere):")
print(np.eye(4))

print("
arange — like range() but returns array:")
print(np.arange(0, 10, 2))    # start, stop (exclusive), step

print("
linspace — evenly spaced values, you control HOW MANY:")
print(np.linspace(0, 1, 6))   # 6 values from 0 to 1 inclusive

print("
full — fill with a specific value:")
print(np.full((2, 3), 7))

print("
reshape — change shape without changing data:")
flat = np.arange(12)
matrix = flat.reshape(3, 4)
print(flat)
print(matrix)

## 10.3 Indexing and Slicing

**Indexing in NumPy** works like Python lists, extended to multiple dimensions. For a 2D array `arr`, `arr[row, col]` selects a single element.

**Slicing** uses the same `start:stop:step` syntax, applied per dimension: `arr[row_slice, col_slice]`.

**Boolean indexing** is one of NumPy's most powerful features. Instead of an index number, you provide an array of `True`/`False` values — NumPy returns only the elements where the value is `True`. This is how you filter data without a loop, and it is used constantly in data cleaning and preprocessing.

In [ ]:
import numpy as np

arr = np.arange(20).reshape(4, 5)
print("Array:")
print(arr)
print()

# Single element — [row, col]
print("arr[0, 0] =", arr[0, 0])    # top-left
print("arr[2, 3] =", arr[2, 3])    # row 2, col 3
print("arr[-1, -1] =", arr[-1,-1]) # bottom-right

print()
# Slicing — works per dimension
print("First 2 rows:")
print(arr[:2, :])       # rows 0-1, all columns

print("Columns 1 to 3:")
print(arr[:, 1:4])      # all rows, columns 1-3

print("Submatrix (rows 1-2, cols 2-3):")
print(arr[1:3, 2:4])

In [ ]:
import numpy as np

# Boolean indexing — filter without a loop
# This is fundamental to data science
data = np.array([15, 42, 8, 73, 29, 91, 5, 67])

# Step 1: Create a boolean mask
mask = data > 30
print("Data:", data)
print("Mask (>30):", mask)   # array of True/False

# Step 2: Use the mask to filter
filtered = data[mask]
print("Values > 30:", filtered)

# One-liner version (most common in practice)
print("Values > 30:", data[data > 30])
print("Values between 20 and 70:", data[(data >= 20) & (data <= 70)])

# np.where — replace values based on condition
# np.where(condition, value_if_true, value_if_false)
capped = np.where(data > 50, 50, data)   # cap everything above 50 at 50
print("Capped at 50:", capped)

## 10.4 Vectorised Operations and Broadcasting

**Vectorised operations** apply a function or operator to every element simultaneously, without a loop. The operation is implemented in C, making it orders of magnitude faster than a Python loop.

**Broadcasting** is NumPy's intelligent way of handling operations between arrays of different shapes. Instead of requiring both arrays to be the same size, NumPy automatically "stretches" the smaller array to match the shape of the larger one — without copying data.

**Broadcasting rules (simplified):**
When operating on two arrays, NumPy compares their shapes from right to left. Two dimensions are compatible if they are equal OR one of them is 1. A size-1 dimension gets stretched to match the other.

In [ ]:
import numpy as np

a = np.array([1, 2, 3, 4, 5])
b = np.array([10, 20, 30, 40, 50])

# All basic operations work element-wise
print("a + b =", a + b)
print("a * b =", a * b)
print("a ** 2 =", a ** 2)

# NumPy maths functions also work element-wise
print("sqrt(b) =", np.sqrt(b))
print("log(b) =", np.log(b).round(2))

print()
# Broadcasting — add a scalar to every element
print("a + 100 =", a + 100)

# Add a 1D array to every row of a 2D matrix
matrix = np.array([[1, 2, 3],
                   [4, 5, 6],
                   [7, 8, 9]])   # shape (3, 3)
row    = np.array([10, 20, 30])  # shape (3,)

# NumPy broadcasts 'row' to match matrix shape
print("matrix + row (broadcast):")
print(matrix + row)   # row [10,20,30] is added to EACH row of matrix

In [ ]:
import numpy as np

# Aggregation — compute summary statistics over entire array or per axis
data = np.array([[1, 2, 3],
                 [4, 5, 6],
                 [7, 8, 9]])

print("Sum of all elements:", data.sum())
print("Maximum value:", data.max())
print("Mean:", data.mean())

# axis=0 means "collapse rows" -> one result per COLUMN
print("Column sums (axis=0):", data.sum(axis=0))    # [12, 15, 18]
print("Column means (axis=0):", data.mean(axis=0))

# axis=1 means "collapse columns" -> one result per ROW
print("Row sums (axis=1):", data.sum(axis=1))       # [6, 15, 24]
print("Row means (axis=1):", data.mean(axis=1))

# Standard deviation and variance
print("Std deviation:", data.std().round(3))

## 10.5 Linear Algebra

**Why linear algebra for ML?**
Machine learning IS linear algebra. When a neural network processes an input:
- The input is a vector (1D array)
- Each layer applies a matrix multiplication
- Weights are matrices, biases are vectors
- Gradient descent updates these matrices

Understanding matrix operations means understanding what your ML model is actually doing.

**Key operations:**
- Matrix multiplication `@` — the core operation of neural networks
- Transpose `.T` — flip rows and columns
- Inverse `np.linalg.inv()` — divide by a matrix (used in linear regression's closed-form solution)
- Eigenvalues/eigenvectors — fundamental to PCA (dimensionality reduction)

In [ ]:
import numpy as np

A = np.array([[1, 2],
              [3, 4]])
B = np.array([[5, 6],
              [7, 8]])

print("A:")
print(A)
print("B:")
print(B)

# Matrix multiplication — NOT element-wise, it's the dot product
print("
A @ B (matrix multiply):")
print(A @ B)   # or np.matmul(A, B)

# Transpose — flip rows and columns
print("
Transpose of A:")
print(A.T)

# Determinant — a scalar that summarises a matrix
print("
det(A):", np.linalg.det(A))

# Inverse — A @ inv(A) = identity matrix
A_inv = np.linalg.inv(A)
print("
Inverse of A:")
print(A_inv.round(3))

print("
A @ inv(A) (should be identity):")
print((A @ A_inv).round(10))

## 10.6 Random Numbers

**Why random numbers in ML?**
- Weight initialisation in neural networks
- Train/test splits and cross-validation shuffles
- Data augmentation (randomly flip, rotate, crop images)
- Monte Carlo simulations
- Bootstrap sampling for confidence intervals

**Always set a seed for reproducibility:**
Random numbers in computers are actually "pseudo-random" — generated by a deterministic algorithm. Setting a seed ensures you get the SAME sequence every time, so your experiments are reproducible.

In [ ]:
import numpy as np

# Modern API: use a Generator with an explicit seed
rng = np.random.default_rng(seed=42)   # seed=42 means reproducible results

# Uniform distribution — values equally likely between 0 and 1
uniform = rng.uniform(0, 1, size=5)
print("Uniform:", uniform.round(3))

# Normal (Gaussian) distribution — bell curve
# loc=mean, scale=standard deviation
normal = rng.normal(loc=0, scale=1, size=5)
print("Normal:", normal.round(3))

# Random integers
integers = rng.integers(1, 101, size=10)   # 1 to 100 inclusive
print("Integers:", integers)

# Shuffle — randomise order in place
arr = np.arange(1, 11)
rng.shuffle(arr)
print("Shuffled:", arr)

# Choice — random sample from an array
choices = rng.choice(arr, size=5, replace=False)   # no repeats
print("Sample of 5:", choices)

# Simulating ML weight initialisation (Xavier/Glorot initialisation)
input_size, output_size = 128, 64
limit = np.sqrt(6 / (input_size + output_size))
weights = rng.uniform(-limit, limit, size=(input_size, output_size))
print(f"
Weight matrix shape: {weights.shape}")
print(f"Weight range: [{weights.min():.4f}, {weights.max():.4f}]")

---

## Key Takeaways

- NumPy arrays are typed, fixed-size, and 10-100x faster than Python lists for maths
- **Never loop over array elements** — use vectorised operations instead
- Slicing syntax: `arr[row_start:row_stop, col_start:col_stop]`
- **Boolean indexing** lets you filter rows without a loop — essential for data cleaning
- **Broadcasting** automatically applies operations across dimensions
- `axis=0` collapses rows (result per column); `axis=1` collapses columns (result per row)
- Always use `np.random.default_rng(seed=42)` for reproducible experiments

## Exercises

[10_exercises.ipynb](exercises/10_exercises.ipynb) | [10_solutions.ipynb](exercises/10_solutions.ipynb)

## Next: [11 — Pandas](../11_pandas/11_pandas.ipynb)
